In [4]:
# pip install pyspark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Temurin-17.0.19+10 (build 17.0.19+10)
OpenJDK 64-Bit Server VM Temurin-17.0.19+10 (build 17.0.19+10, mixed mode, sharing)



In [8]:
import pyspark
print(pyspark.__version__)

4.1.1


In [1]:
import subprocess
result = subprocess.run(["java", "-version"], capture_output=True, text=True)
print(result.stderr)

import pyspark
print(pyspark.__version__)

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "D:/pyspark_udemy_codespace/setup/spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Temurin-17.0.19+10 (build 17.0.19+10)
OpenJDK 64-Bit Server VM Temurin-17.0.19+10 (build 17.0.19+10, mixed mode, sharing)

3.5.1
Spark version: 3.5.3


In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS spark_db")
spark.sql("USE spark_db")
print("Database created OK")

Database created OK


In [ ]:
spark.sql("DROP TABLE IF EXISTS spark_db.diamonds")

import os, shutil
diamonds_path = 'D:/pyspark_udemy_codespace/setup/spark-warehouse/spark_db.db/diamonds'
if os.path.exists(diamonds_path):
    shutil.rmtree(diamonds_path, ignore_errors=True)

spark.sql("""
    CREATE TABLE spark_db.diamonds(
          carat DOUBLE,
          clarity STRING,
          color STRING,
          cut STRING,
          depth STRING,
          price DOUBLE
    )
""")

print("Table created OK")

Table created OK


In [ ]:
# Check the first few lines of the file
with open("D:/pyspark_udemy_codespace/data/diamonds.json", "r") as f:
    for i, line in enumerate(f):
        print(line)
        if i > 3:
            break

{"cut":"Ideal","color":"E","clarity":"SI2","carat":0.23,"depth":61.5,"price":326}

{"cut":"Premium","color":"E","clarity":"SI1","carat":0.21,"depth":59.8,"price":326}

{"cut":"Good","color":"E","clarity":"VS1","carat":0.23,"depth":56.9,"price":327}

{"cut":"Premium","color":"I","clarity":"VS2","carat":0.29,"depth":62.4,"price":334}

{"cut":"Good","color":"J","clarity":"SI2","carat":0.31,"depth":63.3,"price":335}



In [ ]:
import os

DATA_PATH = os.path.expanduser("D:/pyspark_udemy_codespace/data/diamonds.json")

df = spark.read.json(DATA_PATH)
df.write.mode("overwrite").saveAsTable("spark_db.diamonds")

print("Rows Loaded: ", df.count())
df.show(5)

Rows Loaded:  53940
+-----+-------+-----+-------+-----+-----+
|carat|clarity|color|    cut|depth|price|
+-----+-------+-----+-------+-----+-----+
| 0.23|    SI2|    E|  Ideal| 61.5|  326|
| 0.21|    SI1|    E|Premium| 59.8|  326|
| 0.23|    VS1|    E|   Good| 56.9|  327|
| 0.29|    VS2|    I|Premium| 62.4|  334|
| 0.31|    SI2|    J|   Good| 63.3|  335|
+-----+-------+-----+-------+-----+-----+
only showing top 5 rows


In [7]:
result = spark.sql("""
    select color, avg(price) as avg_price
    from spark_db.diamonds
    group by color
    order by avg_price desc
""")

result.show()

+-----+------------------+
|color|         avg_price|
+-----+------------------+
|    J|  5323.81801994302|
|    I| 5091.874953891553|
|    H| 4486.669195568401|
|    G| 3999.135671271697|
|    F| 3724.886396981765|
|    D|3169.9540959409596|
|    E|3076.7524752475247|
+-----+------------------+



In [8]:
spark.conf.get("spark.sql.warehouse.dir")

'file:/workspaces/pyspark_udemy_codespace/setup/spark-warehouse'

In [9]:
# Check what tables actually exist
spark.sql("SHOW TABLES IN spark_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
| spark_db| diamonds|      false|
+---------+---------+-----------+



In [10]:
df = spark.table("spark_db.diamonds")
df.show()

+-----+-------+-----+---------+-----+-----+
|carat|clarity|color|      cut|depth|price|
+-----+-------+-----+---------+-----+-----+
| 0.23|    SI2|    E|    Ideal| 61.5|  326|
| 0.21|    SI1|    E|  Premium| 59.8|  326|
| 0.23|    VS1|    E|     Good| 56.9|  327|
| 0.29|    VS2|    I|  Premium| 62.4|  334|
| 0.31|    SI2|    J|     Good| 63.3|  335|
| 0.24|   VVS2|    J|Very Good| 62.8|  336|
| 0.24|   VVS1|    I|Very Good| 62.3|  336|
| 0.26|    SI1|    H|Very Good| 61.9|  337|
| 0.22|    VS2|    E|     Fair| 65.1|  337|
| 0.23|    VS1|    H|Very Good| 59.4|  338|
|  0.3|    SI1|    J|     Good| 64.0|  339|
| 0.23|    VS1|    J|    Ideal| 62.8|  340|
| 0.22|    SI1|    F|  Premium| 60.4|  342|
| 0.31|    SI2|    J|    Ideal| 62.2|  344|
|  0.2|    SI2|    E|  Premium| 60.2|  345|
| 0.32|     I1|    E|  Premium| 60.9|  345|
|  0.3|    SI2|    I|    Ideal| 62.0|  348|
|  0.3|    SI1|    J|     Good| 63.4|  351|
|  0.3|    SI1|    J|     Good| 63.8|  351|
|  0.3|    SI1|    J|Very Good| 